In [1]:

from pathlib import Path
import pandas as pd

BASE = Path(r"D:\RouteRanker\data\raw\mumbai")
matches = list(BASE.rglob("routes.txt"))

if not matches:
    print("routes.txt not found. Check the extracted folder.")
else:
    GTFS_DIR = matches[0].parent
    data = {}

    for name in ["routes", "stops", "trips", "stop_times"]:
        path = GTFS_DIR / f"{name}.txt"

        if path.exists():
            data[name] = pd.read_csv(path, dtype=str)
            df = data[name]
            print(f"\n{name}.txt: {len(df)} rows, {len(df.columns)} columns")
            print("Missing values:", int(df.isna().sum().sum()))
        else:
            print(f"{name}.txt not found")



routes.txt: 1609 rows, 5 columns
Missing values: 0

stops.txt: 10279 rows, 4 columns
Missing values: 0

trips.txt: 99080 rows, 5 columns
Missing values: 0

stop_times.txt: 2655026 rows, 6 columns
Missing values: 0


In [2]:

for name, col in [
    ("routes", "route_id"),
    ("stops", "stop_id"),
    ("trips", "trip_id")
]:
    if name in data and col in data[name].columns:
        print(f"{name} duplicate IDs:",
              data[name][col].duplicated().sum())

if all(k in data for k in ["routes", "trips"]):
    print("Trips with missing routes:",
          (~data["trips"]["route_id"].isin(
              data["routes"]["route_id"])).sum())

if all(k in data for k in ["trips", "stop_times"]):
    print("Stop times with missing trips:",
          (~data["stop_times"]["trip_id"].isin(
              data["trips"]["trip_id"])).sum())

if all(k in data for k in ["stops", "stop_times"]):
    print("Stop times with missing stops:",
          (~data["stop_times"]["stop_id"].isin(
              data["stops"]["stop_id"])).sum())


routes duplicate IDs: 0
stops duplicate IDs: 0
trips duplicate IDs: 0
Trips with missing routes: 0
Stop times with missing trips: 0
Stop times with missing stops: 0


In [3]:

from pathlib import Path
import pandas as pd

BASE = Path(r"D:\RouteRanker\data\raw\mumbai")
GTFS_DIR = next(BASE.rglob("routes.txt")).parent

for path in sorted(GTFS_DIR.glob("*.txt")):
    df = pd.read_csv(path, dtype=str)
    print(f"\n{path.name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", list(df.columns))
    print("Missing values:", df.isna().sum().to_dict())



agency.txt
Rows: 4
Columns: 6
Column names: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_phone', 'agency_email']
Missing values: {'agency_id': 0, 'agency_name': 0, 'agency_url': 0, 'agency_timezone': 0, 'agency_phone': 1, 'agency_email': 2}

calendar.txt
Rows: 1
Columns: 10
Column names: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Missing values: {'service_id': 0, 'monday': 0, 'tuesday': 0, 'wednesday': 0, 'thursday': 0, 'friday': 0, 'saturday': 0, 'sunday': 0, 'start_date': 0, 'end_date': 0}

feed_info.txt
Rows: 1
Columns: 6
Column names: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_contact_url']
Missing values: {'feed_publisher_name': 0, 'feed_publisher_url': 0, 'feed_lang': 0, 'feed_start_date': 0, 'feed_end_date': 0, 'feed_contact_url': 0}

routes.txt
Rows: 1609
Columns: 5
Column names: ['route_id', 'agency_id', 'route_short_na

In [4]:

# Check relationships between Mumbai GTFS files

routes = data["routes"]
stops = data["stops"]
trips = data["trips"]
stop_times = data["stop_times"]

print("Trips referencing missing routes:",
      (~trips["route_id"].isin(routes["route_id"])).sum())

print("Stop times referencing missing trips:",
      (~stop_times["trip_id"].isin(trips["trip_id"])).sum())

print("Stop times referencing missing stops:",
      (~stop_times["stop_id"].isin(stops["stop_id"])).sum())

print("Trips without stop-time records:",
      (~trips["trip_id"].isin(stop_times["trip_id"])).sum())

if "service_id" in trips.columns:
    print("\nService ID column exists in trips.txt")
    print("Unique service IDs:", trips["service_id"].nunique())


Trips referencing missing routes: 0
Stop times referencing missing trips: 0
Stop times referencing missing stops: 0
Trips without stop-time records: 0

Service ID column exists in trips.txt
Unique service IDs: 1



## Dataset Relationships

This dataset follows the General Transit Feed Specification (GTFS), which organizes public transit information into related files.

- **routes.txt:** Contains route information. Routes are identified by `route_id`.
- **trips.txt:** Contains scheduled trips. The `route_id` connects a trip to a route, while `service_id` identifies its service schedule.
- **stop_times.txt:** Contains scheduled stop times and stop order for each trip. The `trip_id` connects records to trips, and `stop_id` connects them to stops.
- **stops.txt:** Contains stop identifiers, names, and available location information.
- **calendar.txt:** When present, defines regular service days and operating periods.
- **calendar_dates.txt:** When present, specifies additions or exceptions to regular service dates.

### Relationship Validation

The notebook checks whether trips reference valid routes, stop-time records reference valid trips and stops, and trips have stop-time records. Duplicate identifiers and missing values are also examined.
